# Planning:


### Required Data:
Profiles can be nodes in a graph, each containing the necessary data. They will each have a user id that can be accessed through a dictionary.
Connections, blocked users, and skills can be stored under the profile node, in a set.
User id, name, job title, employer, industry, and location can simply be variables under each profile node.
A disconnected group, an isolated user, and at least one blocked pair can be created when generating the graph.

### Required Data Structures:
Dictionary will be used when storing / searching for user ids.
Sets will be used for connections, blocked users, and skills.
Adjacency List Graph will be used for the profiles and connections.
Queue can be used to implement breadth first search for if two users are connected.
Heap can be used to implement / store top connection recommendations
**Note, the required extension should use a data structure other than the above five in some justifiable way**





|Feature or Operation|Candidate Structures|Selected Structure|Reason|Expected Complexity|Important Tradeoff|
|:------------------:|:------------------:|:----------------:|:----:|:-----------------:|:----------------:|
|Profile Lookup      |List, dictionary, tree| Dictionary | Fast lookup by user ID | O(1) | Additional Memory|
|Mutual Connections or Skills  |List, set |Set | Fast Intersection| Expected O(n) where n is the smaller of the two sets| Unordered Connections|
|Professional Connections| Adjacency List Graph, Adjacency Matrix Graph | Adjacency List Graph| Smaller storage, quicker addition of vertex and edges|To add vertex or edge: O(1)| Harder to see if two vertices are adjacent, and thus slower to search the graph|
|Breadth First Search (As in shortest_connection_chain and find_network_components)| Really only Queue| Queue| The other option would be depth first searching (and thus a stack), but this is inefficient for finding the shortest chain, as it is likely to do extra work before finding a shorter route| O(V + E) where V is vertices and E is edges| The algorithm may do extra work if the degree of separation between two individuals is especially high|
|Top candidate recommendations| Heap, list, dictionary| Heap| Easy access of largest value| O(1) to find top candidate, but O(n + h) to insert new heap node, where n is the current amount of heap nodes and h is the height of the graph| Can be a little costly to create initially|
|**Extension**|**Extension Candidates**|**Extension Selection**|**Reason**|**Expected Complexity**|**Important Tradeoff**|

#Diagram of recommendation request data structures

```mermaid
flowchart TD
A[User Dictionary] -->|Input For| B[Recommendation Request]
B-->|Compares via|C[Connections and Skills Sets]
C-->|Stores Score in|D[Candidate Score Heap]
D -->|Outputs Final Recommendations|B
```

In [129]:
import random
random.seed(2112)
import math

In [130]:
class Profile:

    def __init__(self, user_id, name, location, job_title = "Unemployed", employer = "None", industry = "None"):

        self.user_id = user_id
        self.name = name
        self.location = location
        self.job_title = job_title
        self.employer = employer
        self.industry = industry
        self.connections = set()
        self.blocked_users = set()
        self.skills = set()

    def __str__(self):

        return f"User #{self.user_id}: {self.name} lives in {self.location} and works as a {self.job_title} for {self.employer} in the {self.industry} industry.\nThey have the following skills: {str(self.skills)}"

    def connection_print(self):

        to_print = f"Name: {self.name}. Connections: | "

        for c in self.connections:
            to_print += f"{c.name} | "

        print(to_print)
        return None


In [131]:
#Added only for heap usage in recommendations
class HeapNode:

    def __init__(self, value, user, depth):

        self.value = value
        self.user = user
        self.left = None
        self.right = None
        self.parent = None
        self.max_depth = 0
        self.depth = depth

    def __str__(self):

        queue = [self]
        to_print = ""

        while len(queue) > 0:
            current = queue[0]
            to_print += f"\n{str(current.value)} "
            if current.left:
                to_print += f"with children {current.left.value} "
                queue.append(current.left)
                if current.right:
                    to_print += f"and {current.right.value}"
                    queue.append(current.right)
            queue.pop(0)

        return to_print

    def insert (self, new_value, new_user):

        def find_open_spot(self):

            if self.max_depth == 0:
                spot = (self,True)

            else:
                temp = self
                while temp.right:
                    temp = temp.right
                if temp.depth == self.max_depth:
                    candidate_depth = self.max_depth
                else:
                    candidate_depth = self.max_depth - 1
                candidates = []
                queue = [self]

                while len(queue) > 0:

                    current_node = queue[0]

                    if current_node.depth == candidate_depth:
                        candidates.append(current_node)

                    else:
                        if current_node.left:
                            queue.append(current_node.left)
                        if current_node.right:
                            queue.append(current_node.right)

                    queue.pop(0)


                for i in range(len(candidates)):

                    c = candidates[i]
                    if not (c.left and c.right):
                        if not c.left:
                            spot = (c,True)
                        else:
                            spot = (c,False)
                        break
            return spot

        #Swaps the node up the tree until its parent is larger or its parent is tied and the parent's ID is larger
        def swappify(just_added):

            current = just_added


            while current.parent.value < current.value:

                bigger = current.value
                bigger_user = current.user
                current.value = current.parent.value
                current.user = current.parent.user
                current.parent.value = bigger
                current.parent.user = bigger_user
                current = current.parent
                if current.parent == None:
                    break

            if current.parent:
                while current.parent.value == current.value and current.parent.user.user_id < current.user.user_id:

                    bigger_user = current.user
                    current.user = current.parent.user
                    current.parent.user = bigger_user
                    current = current.parent
                    if current.parent == None:
                        break

            return current

        def reconfigure_depth(node):

            current = node
            while current.parent:

                if current.parent.max_depth == current.max_depth:
                    current.parent.max_depth += 1
                current = current.parent
            return None
                

        spot = find_open_spot(self)

        new_parent = spot[0]
        new_depth = new_parent.depth + 1

        if spot[1] == True:
            new_parent.left = HeapNode(new_value,new_user,new_depth)
            new_parent.left.parent = new_parent
            new_parent.left.max_depth = 0

            reconfigure_depth(new_parent.left)

            newly_added = new_parent.left

        else:
            new_parent.right = HeapNode(new_value,new_user,new_depth)
            new_parent.right.parent = new_parent
            new_parent.right.max_depth = 0

            newly_added = new_parent.right

        final_position = swappify(newly_added)
        return final_position

    def extract_max(self):

        def find_last(self):

            current = self
            already_checked = set()

            while current.right:
                current = current.right

            while current.depth != self.max_depth:
                already_checked.add(current)
                while current in already_checked:
                    if current.parent.right == current:
                        current = current.parent.left
                        while current.right:
                            current = current.right
                        if current.left:
                            current = current.left
                    else:
                        current = current.parent

            return current

        #Swaps the node down the tree until both its children are smaller or one / both of its children are tied AND those children's IDs are smaller
        def swappify(self):

            current = self

            while current.value < current.left.value or current.value < current.right.value:
                if current.right.value > current.left.value:
                    smaller = current.value
                    smaller_user = current.user
                    current.value = current.right.value
                    current.user = current.right.user
                    current.right.value = smaller
                    current.right.user = smaller_user
                    current = current.right
                else:
                    smaller = current.value
                    smaller_user = current.user
                    current.value = current.left.value
                    current.user = current.left.user
                    current.left.value = smaller
                    current.left.user = smaller_user
                    current = current.left
                if current.right == None:
                    break

            if current.right == None:
                if current.left:
                    if current.left.value > current.value:
                        smaller = current.value
                        smaller_user = current.user
                        current.value = current.left.value
                        current.user = current.left.user
                        current.left.value = smaller
                        current.left.user = smaller_user
                        current = current.left

            if current.left and current.right:
                #While one of the children have the same value and that child has a larger ID
                while (current.value == current.left.value or current.value == current.right.value):

                    if current.value == current.left.value and current.value == current.right.value:
                        if current.user.user_id < current.right.user.user_id or current.user.user_id < current.left.user.user_id:
                            if current.right.user.user_id > current.left.user.user_id:
                                smaller_user = current.user
                                current.user = current.right.user
                                current.right.user = smaller_user
                                current = current.right
                            else:
                                smaller_user = current.user
                                current.user = current.left.user
                                current.left.user = smaller_user
                                current = current.left
                            if current.right == None:
                                break
                        else:
                            break

                    elif current.value == current.right.value:
                        if current.user.user_id < current.right.user.user_id:
                            smaller_user = current.user
                            current.user = current.right.user
                            current.right.user = smaller_user
                            current = current.right
                            if current.right == None:
                                break
                        else:
                            break

                    else:
                        if current.user.user_id < current.left.user.user_id:
                            smaller_user = current.user
                            current.user = current.right.user
                            current.right.user = smaller_user
                            current = current.right
                            if current.right == None:
                                break
                        else:
                            break

            if current.right == None:
                if current.left:
                    if current.user.user_id < current.left.user.user_id:
                        smaller_user = current.user
                        current.user = current.left.user
                        current.left.user = smaller_user
                        current = current.left
 
            return None

        max_value = self.value
        max_user = self.user

        last = find_last(self)
        self.value = last.value
        self.user = last.user

        if last.parent:
            if last.parent.right:
                last.parent.right = None
            else:
                last.parent.left = None

        swappify(self)

        return (max_value,max_user)

        

In [132]:
#Generic Functions
def shortest_connection_chain(start_user, target_user, print_it = False):

    if start_user == target_user:
        if print_it == True:
            print(f"That's the same user.")
        return((0,[start_user.name]))

    queue = []
    #Uses tuple to store the user Profile and the names of connections along the way
    current_node_path = (start_user,[start_user.name])
    queue.append(current_node_path)

    visited = set()

    found = False

    #Breadth First Search Loop
    while len(queue) != 0:

        #splits tuple into component parts for readability
        current_node = current_node_path[0]
        current_path = current_node_path[1]

        for c in current_node.connections:

            #This connection is the one we're looking for
            if c == target_user:
                current_path.append(c.name)
                found = True
                break

            #This connection needs to be searched later
            if c not in visited:
                new_path = current_path.copy()
                new_path.append(c.name)
                if (c,new_path) not in queue:
                    queue.append((c,new_path))

        #Prints / Returns the depth and path if we found the correct connection
        if found == True:
            depth = len(current_path) - 1
            if print_it == True:
                print(f"The shortest path between {start_user.name} and {target_user.name} is {str(depth)} connections. The path is {str(current_path)}")
            return((depth,current_path))

        #Only activates if the correct connection was not found. Remove from queue, add to visited, and change current_node_path
        visited.add(current_node)
        queue.pop(0)
        if len(queue) != 0:
            current_node_path = queue[0] 

    #We only get here if length of queue is 0, meaning no connection could be found
    if print_it == True:
        print(f"{start_user.name} and {target_user.name} are not connected.")
    return((None,None))

def connect(user,friends_list):

    for f in friends_list:
        user.connections.add(f)
        f.connections.add(user)

    return None

def id_name_print(dictionary):
    to_print = ""
    for id,user in dictionary.items():
        to_print += f"User #{str(id)} is {user.name}\n"
    print(to_print)
    return None

def find_network_components(user_dict, print_it = True):

    if not len(user_dict) > 0:
        if print_it == True:
            print("Empty Data Set")
        return None

    user_ids = list(user_dict.keys())
    communities_output = []
    community_count = 0

    while len(user_ids) > 0:

        current_user_id = user_ids[0]
        queue = []
        visited = set()
        queue.append(current_user_id)
        
        community_count += 1
        names = []

        while len(queue) > 0:

            current_user = user_dict[current_user_id]
            user_ids.remove(current_user_id)

            names.append(current_user.name)
            visited.add(current_user_id)

            for c in current_user.connections:

                if c.user_id not in visited:

                    if c.user_id not in queue:
                        queue.append(c.user_id)

            queue.pop(0)
            if len(queue) > 0:
                current_user_id = queue[0]

        communities_output.append((community_count,len(names),names))

    if print_it == True:
        for num, count, names_list in communities_output:

            if count > 1:
                print(f"Community number {str(num)} has {str(count)} people in it. They are:")
                for n in names_list:
                    print(f"\t{n}")
            else:
                print(f"Community number {str(num)} is an isolated user named {names_list[0]}")

    return communities_output

def top_connection_recommendations(user_id,user_dict,k,print_it = True):

    user = user_dict[user_id]
    heap = None

    candidate_ids = list(user_dict.keys())
    candidate_ids.remove(user_id)

    for c in user.connections:
        candidate_ids.remove(c.user_id)

    for b in user.blocked_users:
        candidate_ids.remove(b.user_id)

    if k > len(candidate_ids):
        if print_it == True:
            print("Not enough potential candidates. Try being less connected, or blocking less people")
        return []

    for a in candidate_ids:

        candidate = user_dict[a]
        score = 0

        for c in user.connections:
            if c in candidate.connections:
                score += 4
                break

        for s in user.skills:
            if s in candidate.skills:
                score += 2
                break

        if user.industry == candidate.industry:
            score += 2

        if user.location == candidate.location:
            score += 1

        if heap:
            heap.insert(score,user_dict[a])
        else:
            heap = HeapNode(score,user_dict[a],0)

    if print_it == True:
        print (f"{user.name}'s top {str(k)} recommendations:")

    recommendation_list = []

    for i in range(k):
        score_person = heap.extract_max()
        score = score_person[0]
        person = score_person[1]

        mutual_connections = 0
        mutual_skills = 0
        mutual_industry = False
        mutual_location = False

        for c in user.connections:
            if c in person.connections:
                mutual_connections += 1

        for s in user.skills:
            if s in person.skills:
                mutual_skills += 1

        if user.industry == person.industry:
            mutual_industry = True

        if user.location == person.location:
            mutual_location = True

        to_print = f"Candidate #{str(i+1)}: {person.name} has {str(mutual_connections)} connections in common. "
        if mutual_skills > 0:
            to_print += f"They have {str(mutual_skills)} skills in common. "
        if mutual_industry == True:
            to_print += f"They both work in the {person.industry} industry. "
        if mutual_location == True:
            to_print += f"They both live in {person.location}. "
        to_print += f"Final Score: {str(score)}"

        if print_it == True:
            print(to_print)

        recommendation_list.append((person,score))
    return recommendation_list

        
            
        
        


In [133]:
#Functions for generating dataset
def generate_profiles(number,user_dict):

    #Surnames chosen from most common surnames in the united states according to census.gov
    surnames = ["Smith","Johnson","Williams","Brown","Jones","Garcia","Miller","Rodriguez","Davis","Martinez","Hernandez","Lopez","Gonzalez","Wilson","Anderson"]
    #Given names not chosen that way
    given_names = ["Tom","Tommy","Thomas","Thomathan","Thomicheal","Thomater","Thomeven","Thomephen","Thouglas","Thombert","Orion"]
    #Cities / Locations chosen from the most populous cities in the united states according to census.gov, with the word "city" removed (who calls it 'Chicago city')
    locations = ["New York","Los Angeles","Chicago","Houston","Phoenix"]
    #industries are tuples with the name of the industry and some example jobs in that industry
    industries = [("Construction",["Construction Worker","Construction Manager"]),("Accounting",["Tax Accountant","Auditer","Private Accountant"]),("Tech",["Data Scientist","Software Developer","Obnoxious CEO"])]
    employers = ["Hegemonic Monopoly Inc.","Small Startup and Co."]
    skills = ["Conflict Mitigation","Adaptability","Innovative Thinking","Communication","Strategic Thinking","Customer Service","Creativity","Client Management","Customer Retention","Relationship Management"]
   

    for i in range(number):

        user_id = i+1
        name = f"{random.choice(given_names)} {random.choice(surnames)}"
        location = random.choice(locations)
        industry_list = random.choice(industries)
        industry = industry_list[0]
        job_title = random.choice(industry_list[1])
        employer = random.choice(employers)
        skills_amount = random.randint(1,4)
        profile_skills = random.sample(skills,skills_amount)

        new_node = Profile(user_id,name,location,job_title,employer,industry)
        for skill in profile_skills:
             new_node.skills.add(skill)
        user_dict[user_id] = new_node

    return user_dict

def connect_the_dots(user_dict, minimum_connections = 35, isolated_users = 0, disconnected_triplets = 0):

    def isolate_them(ids):

            for i in range(isolated_users):
                isolated = random.choice(ids)
                ids.remove(isolated)
            return None

    def triplet_disconnection(ids):

        for k in range(disconnected_triplets):
        
                disconnected_ids = random.sample(ids,3)
        
                for y in disconnected_ids:
                    ids.remove(y)
                    friends = []
                    for x in disconnected_ids:
                         if x != y:
                              friends.append(user_dict[x])
                    connect(user_dict[y],friends)
                
        return None

    user_count = len(user_dict)
    minimum_users = isolated_users + (disconnected_triplets * 3)
    possible_connections = math.factorial(user_count - minimum_users) + (disconnected_triplets * 3)
    connection_count = 0

    #checking for errors
    if minimum_connections > possible_connections:
        raise Exception("Not possible to create the minimum number of connections")

    if user_count < minimum_users:
        raise Exception("Not enough users for that many isolated or disconnected groups!")

    user_ids = list(user_dict.keys())

    isolate_them(user_ids)

    triplet_disconnection(user_ids)

    connection_count += disconnected_triplets * 3

    #Stores the user_ids who are not isolated or in a triplet
    social_user_ids = user_ids.copy()

    #Creating a blocked pair
    blocked_pair = random.sample(social_user_ids,2)

    block1 = user_dict[blocked_pair[0]]
    block2 = user_dict[blocked_pair[1]]

    block1.blocked_users.add(block2)
    block2.blocked_users.add(block1)

    #For each id, makes a list of who can't be connected, then randomly assigns some connections
    for id in user_ids:

        profile = user_dict[id]
        cant_friend = {profile}

        for c in profile.connections:

             cant_friend.add(c)

        for c in profile.blocked_users:
             cant_friend.add(c)

        #Note: Potential Friends capped at 3 arbitrarily, to allow for less connections, but could be increased, especially for larger graphs
        number_of_friends = random.randint(0,3)
        friends_list = set()

        while len(friends_list) < number_of_friends:
             potential_friend = user_dict[random.choice(user_ids)]
             if potential_friend not in cant_friend:
                  friends_list.add(potential_friend)
                  cant_friend.add(potential_friend)
                  connection_count += 1
                  
        connect(profile,friends_list)

    #randomly makes connections until minimum connections is satisfied
    while connection_count < minimum_connections:
         
         lonely_node = user_dict[random.choice(social_user_ids)]

         max_connections = len(social_user_ids) - 1 - len(lonely_node.blocked_users)

         if max_connections > len(lonely_node.connections):

            cant_friend = {lonely_node}
            potential_friend = lonely_node

            for friend in lonely_node.connections:
                cant_friend.add(friend)

            for blocked in lonely_node.blocked_users:
                 cant_friend.add(blocked)

            while potential_friend in cant_friend:
                 potential_friend = user_dict[random.choice(social_user_ids)]

            connect(lonely_node,[potential_friend])
            connection_count += 1

    return None
            


In [134]:
user_dict = {}

generate_profiles(30,user_dict)
connect_the_dots(user_dict,35,3,1)

for user in list(user_dict.values()):
    print(user)
    user.connection_print()
    print("\n")

User #1: Tommy Anderson lives in Houston and works as a Auditer for Hegemonic Monopoly Inc. in the Accounting industry.
They have the following skills: {'Client Management'}
Name: Tommy Anderson. Connections: | Thomater Wilson | Thomas Brown | Thomas Rodriguez | Orion Brown | Thomas Gonzalez | 


User #2: Thomephen Garcia lives in Los Angeles and works as a Construction Manager for Small Startup and Co. in the Construction industry.
They have the following skills: {'Customer Retention', 'Client Management', 'Communication'}
Name: Thomephen Garcia. Connections: | 


User #3: Tommy Martinez lives in New York and works as a Obnoxious CEO for Small Startup and Co. in the Tech industry.
They have the following skills: {'Customer Service', 'Adaptability', 'Conflict Mitigation', 'Client Management'}
Name: Tommy Martinez. Connections: | Thomicheal Williams | Thomeven Brown | 


User #4: Thomas Brown lives in Chicago and works as a Construction Manager for Hegemonic Monopoly Inc. in the Constru

In [135]:
#Testing

#First, we'll show the dataset generated according to our specifications.

#Does the dataset contain at least 20 and at most 30 profiles?
#Completes a simple size check on the length of the dataset 

def size_check():
    assert len(user_dict) >= 20,"Error: Less than 20 Users"
    assert len(user_dict) <= 30, "Error: More than 30 users"
    return "Size Check Passed"

#Does the dataset contain at least 35 professional connections?
#Tallies all the connections across all users and divides by 2 (since each connection is represented in a adjacency list twice)

def connection_check():

    connection_count = 0

    #Adds one to connection count for each connection under each user
    for user in list(user_dict.values()):
        for c in user.connections:
            connection_count += 1

    #Divides by 2, since each connection should be represented twice
    connection_count = connection_count / 2

    assert connection_count % 1 == 0, "Error: Not every connection is represented on both sides"
    assert connection_count >= 35, "Error: Too few connections"
    return "Connection Count Check Passed"

#At least 10 skills, 3 industries, four locations, etc.
    #It is actually technically possible for any of these to not be true *within a dataset*, since these are assigned randomly. 
    #I would recommend checking out generate_profiles in functions for generating the dataset.It is easy to verify that there are the correct amount of options for each of these variables.

#At least one disconnected group with three or more users
#Completes the find_network_components task to find isolated communities, then finds the second that has at least three members. If such a community exists, it is isolated from the first community checked, and both have three or more members.

def isolated_triplet_check():

    successful_candidate = None
    second_community = False

    communities = find_network_components(user_dict,False)
    for num,count,names_list in communities:

        if count >= 3:
            #Goes through here the first time it finds a large community
            if second_community == False:
                second_community = True
            #Goes through here the second
            else:
                successful_candidate = (num,names_list)

    assert successful_candidate, "Error: Only one isolated community has three or more members"
    return "Isolated Triplet Check Passed"

#At least one isolated user
#Searches through all users until it finds one with no connections; this is an isolated user

def isolated_user_check():

    isolated_user_bool = False

    for user in list(user_dict.values()):
        #If a user exists with no connections, then there is an isolated user and the check can finish.
        if len(user.connections) == 0:
            isolated_user_bool = True
            break

    assert isolated_user_bool == True, "Error: Every user has at least one connection"
    return "Isolated User Check Passed"

#At least one blocked pair
#Searches through all users until they find one with a block list longer than 0. Then, checks to make sure the block is mutual.

def blocked_user_check():

    user_who_blocks = None

    for user in list(user_dict.values()):
        #Finds one user who has blocked anybody else
        if len(user.blocked_users) > 0:
            user_who_blocks = user
            break

    assert user_who_blocks, "Error, No users have blocked anyone"

    #Fails if the user who we found last step blocked a user that has not blocked them.
    for blocked_user in user_who_blocks.blocked_users:
        assert user_who_blocks in blocked_user.blocked_users, "Error: Only one user has blocked the other"

    return "Blocked User Check Passed"

#Run all tests for dataset:

print("Database Checks:\n")
print(size_check())
print(connection_check())
print(isolated_triplet_check())
print(isolated_user_check())
print(blocked_user_check())
print("\n")

#For cases 1-4, operation 1 (shortest_connection_chain) will be used.

#Case 1: Finding shortest possible path between two directly connected users. Should be 1.
#Finds the first user in the dictionary that has connections. Then, picks a random connection, and runs the shortest_connection_chain on them to verify the answer is 1
#Outputs the names so one can manually check using the list above.

def direct_connection_check():

    start_user = None

    for user in list(user_dict.values()):

        if len(user.connections) > 0:
            start_user = user
            break

    assert start_user, "Error: No user has any connections"

    end_user = next(iter(start_user.connections))

    depth_path = shortest_connection_chain(start_user,end_user)

    depth = depth_path[0]

    assert depth == 1, "Error, Directly connected user not said to be one connection away"
    return f"Direction Connection Check Passed Via Users {start_user.name} and {end_user.name}"

#Case 2: Two users connected through multiple intermediaries. Should be a number greater than 2
#It is technically possible for this not to work, if the dataset is too connected. 
#Still, this finds the largest community of users, and attempts to search for connections between users in that community until one of them returns a connection depth larger than 2.
#It should always work for the default seed of 2112
#It outputs the users and path, which can be checked above.

def indirect_connection_check():

    largest_community_count = 0
    largest_community_num = None
    largest_community_names = []

    communities = find_network_components(user_dict,False)

    #For each distinct community, checks if it is the largest. By the end of this, the largest community should be stored in the variables
    for num,count,name_list in communities:

        if count > largest_community_count:
            largest_community_count = count
            largest_community_num = num
            largest_community_names = name_list

    assert largest_community_num, "Error: No communities found"

    #This will store everyone in the largest community
    connected_users = []

    #Note: This can grab more than just those people in the largest community, if names are shared. We'll account for this larter
    for user in list(user_dict.values()):
        if user.name in largest_community_names:
            connected_users.append(user)

    start_user = None
    end_user = None
    path = None

    for user in connected_users:

        potential_connections = connected_users.copy()
        potential_connections.remove(user)
        for c in user.connections:
            potential_connections.remove(c)

        for connection in potential_connections:
            depth_path = shortest_connection_chain(user,connection)
            #This will not be true if the person is outside the community, solving the problem mentioned earlier
            if depth_path[0]:
                depth = depth_path[0]
                if depth > 2:
                    start_user = user
                    end_user = connection
                    path = depth_path[1]
                    break

    assert start_user, "Error: No two members of the largest community were disconnected enough. Try a different seed"
    return f"Indirect Connection Check Passed. The final connection was between {start_user.name} and {end_user.name}, and the path looked like: {str(path)}"

#Case 3: Two users in disconnected components. Should return (None,None) for the depth and path.
#We'll start by grabbing an entirely isolated user, then trying to connect them to anyone else.
#The check will output the users, so this can be manually checked

def disconnected_check():

    isolated_user = None
    test_user = None

    #finds a user with no connections
    for user in list(user_dict.values()):

        if len(user.connections) == 0:
            isolated_user = user
            break

    assert isolated_user, "Error: No Isolated Users"

    #finds a user with an id different than our other user
    for user in list(user_dict.values()):

        if isolated_user.user_id != user.user_id:
            test_user = user
            break

    assert test_user, "Error: Every user id is the same as the isolated user's. Massive problems abound."

    depth_path = shortest_connection_chain(isolated_user,test_user)

    assert not depth_path[0], "Error: A path was found between disconnected users"
    return f"Disconnected Users Check Passed. There is no connection betwen {isolated_user.name} and {test_user.name}"

#Case 4: A user searched against themself. This should return a depth of 0.
#All that is necessary here is to pick a user and run the check on them.

def self_connection_test():

    test_user_id = next(iter(user_dict))
    test_user = user_dict[test_user_id]

    depth_path = shortest_connection_chain(test_user,test_user)
    assert depth_path[0] == 0, "Error: A path from one user to themselves was not 0"
    return f"Self Connection Test Passed"

#Run all tests for operation 1
print("Operation 1 Tests:\n")
print(direct_connection_check())
print(indirect_connection_check())
print(disconnected_check())
print(self_connection_test())
print("\n")

#For cases 5-7, operation 3 (top_connection_recommendations) will be used

#Case 5: A user with no eligible recommendation. Should return an empty list, meaning not enough candidates were found
#I don't think any users like this exist by default, so we're gonna pick one of the isolated users and ban everyone else. This should create a situation with no recommendations.

def no_recs_check():

    isolated_user = None

    for user in list(user_dict.values()):
        if len(user.connections) == 0:
            isolated_user = user
            break

    assert isolated_user, "Error: No isolated users"

    for user in list(user_dict.values()):
        if user.user_id != isolated_user.user_id:
            if user not in isolated_user.blocked_users:
                isolated_user.blocked_users.add(user)
                user.blocked_users.add(isolated_user)

    assert len(isolated_user.blocked_users) + 1 == len(user_dict), "Error: Isolated User does not have everyone else blocked"

    success_list = top_connection_recommendations(isolated_user.user_id,user_dict,1,False)

    assert len(success_list) == 0, "Error: A reccomendation was found despite everyone being blocked"
    return "No Recommendations Check Passed"

#Case 6: A user with multiple candidates with identical recommendation scores. Should sort higher user ids towards the top of the list.
#We'll run the recommendation function on each user until there is a tie in the top 5. If this happens, we'll check their ids and make sure that they are in the proper order.

def tie_check():

    tied_score = None
    recommendee = None

    for user in list(user_dict.values()):

        list_of_recommendations = top_connection_recommendations(user.user_id,user_dict,5,False)

        if len(list_of_recommendations) > 0:

            last_person = None
            last_score = None
            tied_score = None
            tied_users = None
            
            for person, score in list_of_recommendations:
                if score == last_score:
                    tied_score = score
                    tied_users = (last_person,person)
                    recommendee = user
                    break
                last_person = person
                last_score = score

    assert tied_score, "Error, no two scores tied across the entire dataset"

    first_user_id = tied_users[0].user_id
    second_user_id = tied_users[1].user_id

    assert first_user_id > second_user_id, "Error: User IDs in wrong order"
    return f"Tie Check Passed. For recommendee {recommendee.name}, {tied_users[0].name} and {tied_users[1].name} both had a score of {str(tied_score)}. The id {str(first_user_id)} was larger than id {str(second_user_id)}, which served as a tiebreaker."

#Case 7: A blocked user who would otherwise receive a high score. Function will return a recomendation, then a different user after we block the first recommendation.
#We'll look for any user with at least two recommendations possible. Then, we'll show that the first recommendation given will not appear once that user is blocked.

def blocked_rec_check():

    for user in list(user_dict.values()):

        pair_of_recs = top_connection_recommendations(user.user_id,user_dict,2,False)

        if len(pair_of_recs) > 0:
            final_user = user
            break

    assert len(pair_of_recs) > 0, "Error: No user could generate two recommendations"

    #Gives the profile section of the first recommendation
    top_recommendation = pair_of_recs[0][0]

    assert top_connection_recommendations(final_user.user_id,user_dict,1,False)[0][0] == top_recommendation, "Error: Top recommendation is not top recommendation"

    final_user.blocked_users.add(top_recommendation)

    new_recommendation = top_connection_recommendations(final_user.user_id,user_dict,1,False)[0][0]

    assert new_recommendation != top_recommendation, "Error: Top recommendation remains after being blocked"
    return f"Blocked Recommendation Check Passed. {final_user.name}'s initial recommendation was {top_recommendation.name}. After that user was blocked, their top recommendation was {new_recommendation.name}"

#Run all tests for operation 3
print("Operation 3 tests:\n")
print(no_recs_check())
print(tie_check())
print(blocked_rec_check())
print("\n")

#For case 8, the extension function (**INSERT EXTENSION FUNCTION**) will be used

#INSERT TEST HERE

#Run test for extension
print("Extension test:\n")

Database Checks:

Size Check Passed
Connection Count Check Passed
Isolated Triplet Check Passed
Isolated User Check Passed
Blocked User Check Passed


Operation 1 Tests:

Direction Connection Check Passed Via Users Tommy Anderson and Thomater Wilson
Indirect Connection Check Passed. The final connection was between Thomas Rodriguez and Tommy Anderson, and the path looked like: ['Thomas Rodriguez', 'Thomeven Lopez', 'Thomater Wilson', 'Tommy Anderson']
Disconnected Users Check Passed. There is no connection betwen Thomephen Garcia and Tommy Anderson
Self Connection Test Passed


Operation 3 tests:

No Recommendations Check Passed
Tie Check Passed. For recommendee Thomas Rodriguez, Tommy Jones and Thomater Wilson both had a score of 7. The id 25 was larger than id 16, which served as a tiebreaker.
Blocked Recommendation Check Passed. Tommy Anderson's initial recommendation was Tom Gonzalez. After that user was blocked, their top recommendation was Orion Martinez


Extension test:

